In [1]:
import pandas as pd
import numpy as np
from os import path, listdir
from collections import defaultdict
import alphastats
from alphastats.loader import DIANNLoader
from alphastats.dataset import dataset
import directlfq.lfq_manager as lfq_manager
import scipy as sc
import matplotlib.pyplot as plt
from pyteomics import fasta

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

/home/lerost/.pyenv/versions/3.10.11_alphastats/lib/python3.10/site-packages/outdated/utils.py:14: OutdatedCheckFailedWarning: Failed to check for latest version of package.
Set the environment variable OUTDATED_RAISE_EXCEPTION=1 for a full traceback.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


In [2]:
help(dataset.DataSet.plot_volcano)

Help on function plot_volcano in module alphastats.dataset.dataset:

plot_volcano(self, group1: Union[str, list], group2: Union[str, list], column: str = None, method: str = 'ttest', labels: bool = False, min_fc: float = 1.0, alpha: float = 0.05, draw_line: bool = True, perm: int = 100, fdr: float = 0.05, color_list: list = None)
    Plot Volcano Plot
    
    Args:
        column (str): column name in the metadata file with the two groups to compare
        group1 (str/list): name of group to compare needs to be present in column or list of sample names to compare
        group2 (str/list): name of group to compare needs to be present in column  or list of sample names to compare
        method (str): "anova", "wald", "ttest", "SAM" Defaul ttest.
        labels (bool): Add text labels to significant Proteins, Default False.
        alpha(float,optional): p-value cut off.
        min_fc (float): Minimum fold change.
        draw_line(boolean): whether to draw cut off lines.
        per

In [23]:
!pip list | grep "pyteomics"

pyteomics                 4.7.5


# Fasta

In [5]:
# fasta_path = './fasta/human_ecoli_yeast_13072026.fasta'

# fasta_path = './fasta/sprot_ecoli_ups.fasta'

# i = 0
# test_dct = {'Human':set(), 'Ecoli':set(), 'Yeast':set() , 'Other':set()}
# with open(fasta_path, mode='r') as f:
#     for descr, seq in fasta.read(f) :
#         i += 1
#         if descr.startswith('sp|') :
#             prot_id = descr.split('|')[1]
#         else :
#             prot_id = descr
#         if 'HUMAN' in descr :
#             test_dct['Human'].add(prot_id)
#         elif 'ECOLI' in descr :
#             test_dct['Ecoli'].add(prot_id)
#         elif 'YEAST' in descr :
#             test_dct['Yeast'].add(prot_id)
#         else :
#             test_dct['Other'].add(prot_id)

# for key in test_dct.keys() :
#     print(key, 
#         len(test_dct[key])
#     )
# print('intersection',
#     len(test_dct['Human'] & test_dct['Ecoli']), 
#     len(test_dct['Yeast'] & test_dct['Ecoli']),
#     len(test_dct['Human'] & test_dct['Yeast'])
# )
        # if i >= 40 and i <= 50 :
        #     print(descr)
        # if i >= 300 :
        #     break

In [6]:
fasta_path = './fasta/sprot_ecoli_ups.fasta'

i = 0
ups_ecoli_organism_dict = defaultdict(lambda :'Other')
with open(fasta_path, mode='r') as f:
    for descr, seq in fasta.read(f) :
        if descr.startswith('sp|') :
            prot_id = descr.split('|')[1]
        else :
            prot_id = descr
        if 'HUMAN' in descr :
            ups_ecoli_organism_dict[prot_id] = 'Human'
            ups_ecoli_organism_dict[prot_id.split(' ')[0]] = 'Human'
        elif 'ECOLI' in descr :
            ups_ecoli_organism_dict[prot_id] = 'Ecoli'
        else :
            ups_ecoli_organism_dict[prot_id] = 'Other'
            # ups_ecoli_organism_dict['Other'].add(prot_id)

In [7]:
fasta_path = './fasta/sprot_ecoli_ups_15072026_shuffled.fasta'

i = 0
double_ups_ecoli_organism_dict = defaultdict(lambda :'Other')
with open(fasta_path, mode='r') as f:
    for descr, seq in fasta.read(f) :
        if len(descr.split('|')) > 2 :
            prot_id = descr.split('|')[1]
        else :
            prot_id = descr
        if 'HUMAN' in descr :
            double_ups_ecoli_organism_dict[prot_id] = 'Human'
        elif 'ECOLI' in descr :
            double_ups_ecoli_organism_dict[prot_id] = 'Ecoli'
        else :
            double_ups_ecoli_organism_dict[prot_id] = 'Other'
            # double_ups_ecoli_organism_dict['Other'].add(prot_id)

In [8]:
fasta_path = './fasta/human_ecoli_yeast_13072026.fasta'

h = 0
e = 0
y = 0
o = 0
i = 0
lfqbench_organism_dict = defaultdict(lambda :'Other')
with open(fasta_path, mode='r') as f:
    for descr, seq in fasta.read(f) :
        i += 1
        if len(descr.split('|')) > 2 :
            prot_id = descr.split('|')[1]
        else :
            prot_id = descr
        if 'HUMAN' in descr :
            lfqbench_organism_dict[prot_id] = 'Human'
            lfqbench_organism_dict[prot_id.split(' ')[0]] = 'Human'
            h += 1
        elif 'ECOLI' in descr :
            lfqbench_organism_dict[prot_id] = 'Ecoli'
            e += 1
        elif 'YEAST' in descr :
            lfqbench_organism_dict[prot_id] = 'Yeast'
            y += 1
        else :
            lfqbench_organism_dict[prot_id] = 'Other'
            # lfqbench_organism_dict['Other'].add(prot_id)
            o += 1
        # if 'A0ACI8SQ87' in descr :
        #     print(descr, seq)
print(h, e, y, o)
        # if i >= 40 and i <= 50 :
        #     print(descr, prot_id)
        # if i >= 300 :
        #     break

20652 4403 6066 0


In [9]:
fasta_path = './fasta/human_ecoli_yeast_13072026_shuffled.fasta'

i = 0
double_lfqbench_organism_dict = defaultdict(lambda :'Other')
with open(fasta_path, mode='r') as f:
    for descr, seq in fasta.read(f) :
        if descr.startswith('sp|') :
            prot_id = descr.split('|')[1]
        else :
            prot_id = descr
        if 'HUMAN' in descr :
            double_lfqbench_organism_dict[prot_id] = 'Human'
        elif 'ECOLI' in descr :
            double_lfqbench_organism_dict[prot_id] = 'Ecoli'
        elif 'YEAST' in descr :
            double_lfqbench_organism_dict[prot_id] = 'Yeast'
        else :
            double_lfqbench_organism_dict[prot_id] = 'Other'
            # double_lfqbench_organism_dict['Other'].add(prot_id)

In [10]:
i = 0
h = 0
e = 0
y = 0
for k, v in lfqbench_organism_dict.items() :
    if v == 'Human' :
        h += 1
    elif v == 'Ecoli' :
        e += 1
    elif v == 'Yeast' :
        y += 1
    i ++ 1
print(h, e, y)

20652 4403 6066


In [11]:
# example_input_file_diann = "/path/to/example_input_file_diann.tsv"

# lfq_manager.run_lfq(example_input_file_diann)

# Functions

In [12]:
def alpha_quant(infile, colstarter='', colsplitter='', condstarter='', imputation='mean', organism_dict={}, min_fc=0.5, search_engine='', save_path='') :
    # infile :str - path to report.pg_matrix.tsv from diann
    # colstarter :str - starter of the individual file name
    # colsplitter :str - pattern to split the colname in pair with starter to get condition
    # imputation :str - imputation method
    # min_fc :float - fc threshold
    # search_engine :str - search engine label to store
    
    df = pd.read_csv(infile, sep='\t')
    alphastats_meta_path = infile.replace('pg_matrix.tsv', 'alphastats.meta.tsv')
    all_columns = list(df.columns)
    try :
        samples = sorted([path.basename(col) for col in all_columns if col.startswith(colstarter)], 
                         # path.basename(col)+'_Intensity'
                         key=lambda x: float(x.split(path.basename(colstarter))[-1].split('fmol')[0].replace('_', '.')) + 0.0001*int(x.split('inj')[-1][0]),
                         reverse=False)
        temp_conds = sorted([condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)],
                            key=lambda x: float(x.split('fmol')[0].replace('_', '.')),
                            reverse=False)
        conds = sorted(list(set(temp_conds)), 
                       key=lambda x: float(x.split('fmol')[0].replace('_', '.')),
                       reverse=False)
    except :
        samples = sorted([path.basename(col) for col in all_columns if col.startswith(colstarter)], 
                         reverse=False)
        temp_conds = sorted([condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)],
                            reverse=False)
        conds = sorted(list(set(temp_conds)))
    a = pd.DataFrame([samples, temp_conds], index=['sample', 'condition']).T
    a.to_csv(alphastats_meta_path, sep='\t', index=False)
    
    dia_data = DIANNLoader(file=infile)
    print("Data columns:", dia_data.rawinput.columns.tolist())
    # print("Sample names from loader:", dia_data.sample_names)
    print("Metadata sample names:", a['sample'].tolist())
    my_dataset = dataset.DataSet(
    loader = dia_data,
    metadata_path_or_df=alphastats_meta_path,
    sample_column="sample"
    )
    my_dataset.preprocess(
        remove_contaminations=True,
        subset=True,
        # normalization='quantile',
        imputation=imputation,)
    
    plot = my_dataset.plot_volcano(column = "condition", 
                                group1 = conds[0], 
                                group2 = conds[1], 
                                min_fc = min_fc)
    plot_df = plot.plotting_data
    plot_df.sort_values('pval', inplace=True)
    ll = len(plot_df)
    plot_df['FC_pass'] = np.abs(plot_df['log2fc']) >= min_fc
    bonf = 0.05/ll
    plot_df['Bonf_pass'] = plot_df['pval'] <= bonf
    plot_df['BH_thr'] = [0.05*i/ll for i in range(1, ll+1, 1)]
    plot_df['BH_pass'] = (plot_df['pval'] <= plot_df['BH_thr']).cummin()
    if plot_df['BH_pass'].sum() > 0 :
        border_fdr = plot_df[plot_df['BH_pass']]['pval'].max()
    else :
        border_fdr = 0.05
    # print(plot_df.columns)
    # print(plot_df.head(50))
    
    # break
    plot_df['Organism'] = plot_df['index_'].map(lambda x : list(set([organism_dict[el] for el in x.split(';')]))[0] if len(set([organism_dict[el] for el in x.split(';')]))==1 else 'Contamination').copy()
    plot_df = plot_df.rename(columns={ 'index_':'Protein.Group', })
    plot_df['Human'] = plot_df['Organism'] == 'Human'
    plot_df['Ecoli'] = plot_df['Organism'] == 'Ecoli'
    plot_df['Yeast'] = plot_df['Organism'] == 'Yeast'
    plot_df['FDR_pass'] = plot_df['BH_pass']
    plot_df['Search Engine'] = [search_engine for _ in range(len(plot_df))]
    plot_df['Quantitation'] = ['AlphaPeptStats' for _ in range(len(plot_df))]
    plot_df['Imputation'] = [str(imputation) for _ in range(len(plot_df))]
    plot_df['FC_thr'] = [min_fc for _ in range(len(plot_df))]
    
    if save_path :
        plot_df.to_csv(save_path, index=False, sep='\t')
    else :
        print('not saving results', save_path)
    de = len(plot_df[(plot_df['FDR_pass'] & plot_df['FC_pass'])])
    h_de = len(plot_df[(plot_df['Human'] & plot_df['FDR_pass'] & plot_df['FC_pass'])])
    e_de = len(plot_df[(plot_df['Ecoli'] & plot_df['FDR_pass'] & plot_df['FC_pass'])])
    y_de = len(plot_df[(plot_df['Yeast'] & plot_df['FDR_pass'] & plot_df['FC_pass'])])
    return de, h_de, e_de, y_de, plot_df

In [13]:
def manual_quant(infile, 
                 colstarter='', 
                 colsplitter='', 
                 condstarter='', 
                 imputation='mean', 
                 organism_dict={}, 
                 prot_col='', 
                 min_fc=0.5, 
                 max_nan=3, 
                 search_engine='', 
                 quant='', 
                 save_path='') :
    # infile :str - path to report.pg_matrix.tsv from diann
    # colstarter :str - starter of the individual file name
    # colsplitter :str - pattern to split the colname in pair with starter to get condition
    # max_nan :int - maximum number of nan intensities for 
    # imputation :str - imputation method
    # min_fc :float - fc threshold
    # search_engine :str - search engine label to store
    # quant :str - quantitation label to store
    
    lfq_df = pd.read_csv(infile, sep='\t')
    all_cols = list(lfq_df.columns)
    print(all_cols)
    cols = [col for col in all_cols if col.startswith(colstarter) ]
    conds = sorted(list(set([condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in cols ])))
    cols_A = [col for col in cols if condstarter+col.split(colstarter)[-1].split(colsplitter)[0] == conds[0] ]
    cols_B = [col for col in cols if condstarter+col.split(colstarter)[-1].split(colsplitter)[0] == conds[1] ]
    # print(cols_A, cols_B, sep='\n')
    lfq_df = lfq_df.replace(to_replace={0:np.nan},)
    lfq_df['isna'] = lfq_df[cols].isna().sum(axis=1)
    lfq_df = lfq_df[lfq_df['isna'] <= max_nan].copy()
    for col in cols :
        m = lfq_df[col].min()
        lfq_df[col] = lfq_df[col].replace(to_replace={np.nan:m}).apply(pd.to_numeric, errors='coerce')
    # print(lfq_df[cols].isna().sum())
    lfq_df['pval'] = lfq_df.apply(lambda row: sc.stats.ttest_ind(row[cols_A].to_list(), row[cols_B].to_list(), equal_var=False).pvalue, axis=1)
    lfq_df['log2_FC'] = np.log2(lfq_df[cols_A].mean(skipna=True, axis=1) / lfq_df[cols_B].mean(skipna=True, axis=1))
    ll = len(lfq_df)
    bonf_thr = 0.05/ll
    lfq_df['Bonf_pass'] = lfq_df['pval'] <= bonf_thr
    lfq_df.sort_values('pval', inplace=True)
    lfq_df['BH_thr'] = [0.05*i/ll for i in range(1, ll+1, 1)]
    lfq_df['BH_pass'] = (lfq_df['pval'] <= lfq_df['BH_thr']).cummin()
    lfq_df['FC_pass'] = abs(lfq_df['log2_FC']) > min_fc
    if lfq_df['BH_pass'].sum() > 0 :
        border_fdr = lfq_df[lfq_df['BH_pass']]['pval'].max()
    else :
        border_fdr = 0.05

    if prot_col :
        lfq_df['Organism'] = lfq_df[prot_col].map(lambda x : list(set([organism_dict[el] for el in x.split(';')]))[0] if len(set([organism_dict[el] for el in x.split(';')]))==1 else 'Contamination').copy()
    else :
        lfq_df['Organism'] = lfq_df['Protein.Group'].map(lambda x : list(set([organism_dict[el] for el in x.split(';')]))[0] if len(set([organism_dict[el] for el in x.split(';')]))==1 else 'Contamination').copy()
    lfq_df['Human'] = lfq_df['Organism'] == 'Human'
    lfq_df['Ecoli'] = lfq_df['Organism'] == 'Ecoli'
    lfq_df['Yeast'] = lfq_df['Organism'] == 'Yeast'
    lfq_df['FDR_pass'] = lfq_df['BH_pass']
    lfq_df['Search Engine'] = [search_engine for _ in range(len(lfq_df))]
    lfq_df['Quantitation'] = [quant for _ in range(len(lfq_df))]
    lfq_df['Imputation'] = [imputation for _ in range(len(lfq_df))]
    lfq_df['FC_thr'] = [min_fc for _ in range(len(lfq_df))]

    if save_path :
        lfq_df.to_csv(save_path, index=False, sep='\t')
    else :
        print('not saving results', save_path)
    de = len(lfq_df[(lfq_df['FDR_pass'] & lfq_df['FC_pass'])])
    h_de = len(lfq_df[(lfq_df['Human'] & lfq_df['FDR_pass'] & lfq_df['FC_pass'])])
    e_de = len(lfq_df[(lfq_df['Ecoli'] & lfq_df['FDR_pass'] & lfq_df['FC_pass'])])
    y_de = len(lfq_df[(lfq_df['Yeast'] & lfq_df['FDR_pass'] & lfq_df['FC_pass'])])
    return de, h_de, e_de, y_de, lfq_df

In [40]:
lfq_df = pd.read_csv('./search_results/diann231_ups_ecoli/5fmol_vs_0_25fmol/quant_directlfq_min.tsv', sep='\t')
lfq_df.head()

,protein,RD139_Wide_UPS1_0_25fmol_inj1,RD139_Wide_UPS1_0_25fmol_inj2,RD139_Wide_UPS1_0_25fmol_inj3,RD139_Wide_UPS1_5fmol_inj1,RD139_Wide_UPS1_5fmol_inj2,RD139_Wide_UPS1_5fmol_inj3,isna,pval,log2_FC,Bonf_pass,BH_thr,BH_pass,FC_pass,Organism,Human,Ecoli,Yeast,FDR_pass,Search Engine,Quantitation,Imputation,FC_thr
0,P67662,141297.235101,133796.22389,115453.070052,9.217357e+05,9.697915e+05,9.457159e+05,3,0.000011,-2.860923,True,0.000024,True,True,Ecoli,False,True,False,True,diann231,directlfq,min,1.660964
1,O00762ups|UBE2C_HUMAN_UPS,141297.235101,133796.22389,115453.070052,1.363752e+07,1.387897e+07,1.397899e+07,3,0.000050,-6.731316,False,0.000049,False,True,Human,True,False,False,False,diann231,directlfq,min,1.660964
2,P02787ups|TRFE_HUMAN_UPS,141297.235101,133796.22389,115453.070052,7.026252e+07,7.000102e+07,6.847235e+07,3,0.000064,-9.061969,False,0.000073,True,True,Human,True,False,False,True,diann231,directlfq,min,1.660964
3,P02788ups|TRFL_HUMAN_UPS,141297.235101,133796.22389,115453.070052,9.663253e+07,9.575312e+07,9.845434e+07,3,0.000067,-9.540516,False,0.000098,True,True,Human,True,False,False,True,diann231,directlfq,min,1.660964
4,P01008ups|ANT3_HUMAN_UPS,141297.235101,133796.22389,115453.070052,4.676472e+07,4.669985e+07,4.818103e+07,3,0.000105,-8.502576,False,0.000122,True,True,Human,True,False,False,True,diann231,directlfq,min,1.660964


In [41]:
lfq_df['BH_pass'] = (lfq_df['pval'] <= lfq_df['BH_thr']).cummin()
lfq_df.head()

,protein,RD139_Wide_UPS1_0_25fmol_inj1,RD139_Wide_UPS1_0_25fmol_inj2,RD139_Wide_UPS1_0_25fmol_inj3,RD139_Wide_UPS1_5fmol_inj1,RD139_Wide_UPS1_5fmol_inj2,RD139_Wide_UPS1_5fmol_inj3,isna,pval,log2_FC,Bonf_pass,BH_thr,BH_pass,FC_pass,Organism,Human,Ecoli,Yeast,FDR_pass,Search Engine,Quantitation,Imputation,FC_thr
0,P67662,141297.235101,133796.22389,115453.070052,9.217357e+05,9.697915e+05,9.457159e+05,3,0.000011,-2.860923,True,0.000024,True,True,Ecoli,False,True,False,True,diann231,directlfq,min,1.660964
1,O00762ups|UBE2C_HUMAN_UPS,141297.235101,133796.22389,115453.070052,1.363752e+07,1.387897e+07,1.397899e+07,3,0.000050,-6.731316,False,0.000049,False,True,Human,True,False,False,False,diann231,directlfq,min,1.660964
2,P02787ups|TRFE_HUMAN_UPS,141297.235101,133796.22389,115453.070052,7.026252e+07,7.000102e+07,6.847235e+07,3,0.000064,-9.061969,False,0.000073,False,True,Human,True,False,False,True,diann231,directlfq,min,1.660964
3,P02788ups|TRFL_HUMAN_UPS,141297.235101,133796.22389,115453.070052,9.663253e+07,9.575312e+07,9.845434e+07,3,0.000067,-9.540516,False,0.000098,False,True,Human,True,False,False,True,diann231,directlfq,min,1.660964
4,P01008ups|ANT3_HUMAN_UPS,141297.235101,133796.22389,115453.070052,4.676472e+07,4.669985e+07,4.818103e+07,3,0.000105,-8.502576,False,0.000122,False,True,Human,True,False,False,True,diann231,directlfq,min,1.660964


# Testing

In [31]:
for search_dr in listdir('./search_results') :
    # print(search_dr, '\n')
    if 'LFQbench' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        if search_dr.startswith('fragpipe') :
            workdir = path.join('./search_results', search_dr, 'dia-quant-output')
        else :
            workdir = path.join('./search_results', search_dr)
        # try :
        inpath = path.join(workdir, 'report.pg_matrix.tsv')
        if path.exists(inpath) :
            colstarter = './input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_' # 10fmol_inj2.mzML'
            colsplitter = 'Sample'
            cols = [colstarter+cond+'_Sample_'+s+'_'+rep+'.mzML' for cond in ['A', 'B'] for s in ['Alpha', 'Beta', 'Gamma'] for rep in ['01', '02', '03']]
            df = pd.read_csv(inpath, sep='\t', usecols=cols)
            print(search_dr, len(cols), df.isna().sum().sum(), len(df)*len(cols), round(df.isna().sum().sum()/(len(df)*len(cols)), 3))

fragpipe24_LFQbench_msfragger 18 3896 138366 0.028
fragpipe24_LFQbench_umpire 18 1795 96858 0.019
diann231_LFQbench 18 22019 163296 0.135


In [33]:
for search_dr in listdir('./search_results') :
    # print(search_dr, '\n')
    if 'LFQbench' in search_dr and 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        if search_dr.startswith('fragpipe') :
            workdir = path.join('./search_results', search_dr, 'dia-quant-output')
        else :
            workdir = path.join('./search_results', search_dr)
        # try :
        inpath = path.join(workdir, 'report.pg_matrix.tsv')
        if path.exists(inpath) :
            colstarter = './input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_' # 10fmol_inj2.mzML'
            colsplitter = 'Sample'
            cols = [colstarter+cond+'_Sample_'+s+'_'+rep+'.d' for cond in ['A', 'B'] for s in ['Alpha', 'Beta', 'Gamma'] for rep in ['01', '02', '03']]
            df = pd.read_csv(inpath, sep='\t', usecols=cols)
            print(search_dr, len(cols), df.isna().sum().sum(), len(df)*len(cols), round(df.isna().sum().sum()/(len(df)*len(cols)), 3))

diann231_LFQbenchTOF 18 24614 204930 0.12
fragpipe24_LFQbenchTOF_diatracer 18 3130 187200 0.017


In [45]:
for search_dr in listdir('./search_results') :
    # print(search_dr, '\n')
    if 'ups_ecoli' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        for pair_dr in listdir(path.join('./search_results', search_dr)) :
        # pair_dr = '25fmol_vs_0_25fmol'
            if path.isdir(path.join('./search_results', search_dr, pair_dr)) and '_vs_' in pair_dr :
                conc_1, conc_2 = pair_dr.split('_vs_')
                if search_dr.startswith('fragpipe') :
                    workdir = path.join('./search_results', search_dr, pair_dr, 'dia-quant-output')
                else :
                    workdir = path.join('./search_results', search_dr, pair_dr)
                # try :
                inpath = path.join(workdir, 'report.pg_matrix.tsv')
                if path.exists(inpath) :
                    colstarter = './input_files/PXD026600/RD139_Wide_UPS1_' # 10fmol_inj2.mzML'
                    colsplitter = '_inj'
                    cols = [colstarter+cond+colsplitter+rep+'r.mzML' if cond=='10fmol' and rep=='1' else colstarter+cond+colsplitter+rep+'.mzML' for cond in [conc_1, conc_2] for rep in ['1', '2', '3']]
                    print(search_dr, pair_dr, )#cols)
                    df = pd.read_csv(inpath, sep='\t', usecols=cols)
                    print(df.isna().sum().sum(), len(df)*len(cols), round(df.isna().sum().sum()/(len(df)*len(cols)), 3))


fragpipe24_ups_ecoli_umpire 5fmol_vs_0_25fmol
127 9960 0.013
fragpipe24_ups_ecoli_umpire 25fmol_vs_2_5fmol
23 9990 0.002
fragpipe24_ups_ecoli_umpire 2_5fmol_vs_0_1fmol
69 9696 0.007
fragpipe24_ups_ecoli_umpire 25fmol_vs_1fmol
39 9942 0.004
fragpipe24_ups_ecoli_umpire 50fmol_vs_0_1fmol
111 9618 0.012
fragpipe24_ups_ecoli_umpire 25fmol_vs_0_25fmol
131 10020 0.013
fragpipe24_ups_ecoli_umpire 25fmol_vs_10fmol
23 9906 0.002
fragpipe24_ups_ecoli_umpire 50fmol_vs_10fmol
44 9858 0.004
fragpipe24_ups_ecoli_umpire 10fmol_vs_2_5fmol
45 10062 0.004
fragpipe24_ups_ecoli_umpire 50fmol_vs_25fmol
27 9804 0.003
fragpipe24_ups_ecoli_umpire 10fmol_vs_5fmol
59 9936 0.006
fragpipe24_ups_ecoli_umpire 50fmol_vs_1fmol
62 10032 0.006
fragpipe24_ups_ecoli_umpire 5fmol_vs_1fmol
66 10266 0.006
fragpipe24_ups_ecoli_umpire 1fmol_vs_0_1fmol
38 9780 0.004
fragpipe24_ups_ecoli_umpire 5fmol_vs_2_5fmol
23 9774 0.002
fragpipe24_ups_ecoli_umpire 10fmol_vs_0_25fmol
120 10098 0.012
fragpipe24_ups_ecoli_umpire 50fmol_vs_2_5f

# Flow run

## AlphaPeptStats LFQbench orbitrap imputation RandomForest

In [22]:
imputation = 'knn'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'LFQbench' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr and 'diann231' in search_dr :
        if search_dr.startswith('fragpipe') :
            workdir = path.join('./search_results', search_dr, 'dia-quant-output')
        else :
            workdir = path.join('./search_results', search_dr)
        # try :
        inpath = path.join(workdir, 'report.pg_matrix.tsv')
        if path.exists(inpath) :
            colstarter = './input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_' # 10fmol_inj2.mzML'
            colsplitter = 'Sample'
            save_path = path.join(workdir, 'quant_alphastats_{}.tsv'.format(imputation))
            try :
                se = search_dr.split('LFQbench')[-1].split('_')[1]
            except :
                se = search_dr.split('_')[0]
            de, h_de, e_de, y_de, lfq_df = alpha_quant(inpath, 
                                            colstarter=colstarter, 
                                            colsplitter=colsplitter, 
                                            condstarter='Condition_',
                                            organism_dict=lfqbench_organism_dict, 
                                            imputation=imputation, 
                                            # max_nan=9,
                                            min_fc=0.5, 
                                            search_engine=se, 
                                            # quant='manual',
                                            save_path=save_path)
            print('AlphaStats', search_dr, 'done\n')
        else :
            print('no input file in', search_dr)

2026-08-13 15:09:34,925 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-08-13 15:09:34,930 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-08-13 15:09:34,958 - root - INFO - Contaminations indicated in following columns: ['contamination_library', 'contamination_library'] were removed. In total 8 observations have been removed.
2026-08-13 15:09:34,960 - root - INFO - Imputing data...


diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

diann231_25fmol_vs_5fmol_rerun 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

Data columns: ['Protein.Group', 'Protein.Names', 'Genes', 'First.Protein.Description', 'N.Sequences_Intensity', 'N.Proteotypic.Sequences_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_03.mzML_Intensit

## AlphaPeptStats LFQbench orbitrap imputation RandomForest

In [ ]:
imputation = 'randomforest'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'LFQbench' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        if search_dr.startswith('fragpipe') :
            workdir = path.join('./search_results', search_dr, 'dia-quant-output')
        else :
            workdir = path.join('./search_results', search_dr)
        # try :
        inpath = path.join(workdir, 'report.pg_matrix.tsv')
        if path.exists(inpath) :
            colstarter = './input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_' # 10fmol_inj2.mzML'
            colsplitter = 'Sample'
            save_path = path.join(workdir, 'quant_alphastats_{}.tsv'.format(imputation))
            try :
                se = search_dr.split('LFQbench')[-1].split('_')[1]
            except :
                se = search_dr.split('_')[0]
            de, h_de, e_de, y_de, lfq_df = alpha_quant(inpath, 
                                            colstarter=colstarter, 
                                            colsplitter=colsplitter, 
                                            condstarter='Condition_',
                                            organism_dict=lfqbench_organism_dict, 
                                            imputation=imputation, 
                                            # max_nan=9,
                                            min_fc=0.5, 
                                            search_engine=se, 
                                            # quant='manual',
                                            save_path=save_path)
            print('AlphaStats', search_dr, 'done\n')
        else :
            print('no input file in', search_dr)

2026-07-29 18:45:56,420 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-07-29 18:45:56,427 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-07-29 18:45:56,455 - root - INFO - Contaminations indicated in following columns: ['contamination_library', 'contamination_library'] were removed. In total 9 observations have been removed.
2026-07-29 18:45:56,457 - root - INFO - Imputing data...


diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

Data columns: ['Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes', 'First.Protein.Description', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_03.mz

## AlphaPeptStats LFQbench orbitrap imputation mean

In [ ]:
imputation = 'mean'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'LFQbench' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        if search_dr.startswith('fragpipe') :
            workdir = path.join('./search_results', search_dr, 'dia-quant-output')
        else :
            workdir = path.join('./search_results', search_dr)
        # try :
        inpath = path.join(workdir, 'report.pg_matrix.tsv')
        if path.exists(inpath) :
            colstarter = './input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_' # 10fmol_inj2.mzML'
            colsplitter = 'Sample'
            save_path = path.join(workdir, 'quant_alphastats_{}.tsv'.format(imputation))
            try :
                se = search_dr.split('LFQbench')[-1].split('_')[1]
            except :
                se = search_dr.split('_')[0]
            de, h_de, e_de, y_de, lfq_df = alpha_quant(inpath, 
                                            colstarter=colstarter, 
                                            colsplitter=colsplitter, 
                                            condstarter='Condition_',
                                            organism_dict=lfqbench_organism_dict, 
                                            imputation=imputation, 
                                            # max_nan=9,
                                            min_fc=0.5, 
                                            search_engine=se, 
                                            # quant='manual',
                                            save_path=save_path)
            print('AlphaStats', search_dr, 'done\n')
        else :
            print('no input file in', search_dr)

2026-07-29 18:45:56,420 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-07-29 18:45:56,427 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-07-29 18:45:56,455 - root - INFO - Contaminations indicated in following columns: ['contamination_library', 'contamination_library'] were removed. In total 9 observations have been removed.
2026-07-29 18:45:56,457 - root - INFO - Imputing data...


diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

Data columns: ['Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes', 'First.Protein.Description', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_03.mz

## AlphaPeptStats LFQbench orbitrap imputation None

In [14]:
imputation = None
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'LFQbench' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        if search_dr.startswith('fragpipe') :
            workdir = path.join('./search_results', search_dr, 'dia-quant-output')
        else :
            workdir = path.join('./search_results', search_dr)
        # try :
        inpath = path.join(workdir, 'report.pg_matrix.tsv')
        if path.exists(inpath) :
            colstarter = './input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_' # 10fmol_inj2.mzML'
            colsplitter = 'Sample'
            save_path = path.join(workdir, 'quant_alphastats_{}.tsv'.format(imputation))
            try :
                se = search_dr.split('LFQbench')[-1].split('_')[1]
            except :
                se = search_dr.split('_')[0]
            de, h_de, e_de, y_de, lfq_df = alpha_quant(inpath, 
                                            colstarter=colstarter, 
                                            colsplitter=colsplitter, 
                                            condstarter='Condition_',
                                            organism_dict=lfqbench_organism_dict, 
                                            imputation=imputation, 
                                            # max_nan=9,
                                            min_fc=0.5, 
                                            search_engine=se, 
                                            # quant='manual',
                                            save_path=save_path)
            print('AlphaStats', search_dr, 'done\n')
        else :
            print('no input file in', search_dr)

2026-08-13 14:16:42,334 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-08-13 14:16:42,340 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-08-13 14:16:42,365 - root - INFO - Contaminations indicated in following columns: ['contamination_library', 'contamination_library'] were removed. In total 9 observations have been removed.


diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

Data columns: ['Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes', 'First.Protein.Description', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_03.mz

2026-08-13 14:16:51,948 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-08-13 14:16:51,953 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full


AlphaStats fragpipe24_LFQbench_msfragger done

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

Data columns: ['Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes', 'First.Protein.Description', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_03.m

2026-08-13 14:16:52,132 - root - INFO - Contaminations indicated in following columns: ['contamination_library', 'contamination_library'] were removed. In total 7 observations have been removed.


DataSet has been created.


2026-08-13 14:16:58,322 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-08-13 14:16:58,328 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-08-13 14:16:58,358 - root - INFO - Contaminations indicated in following columns: ['contamination_library', 'contamination_library'] were removed. In total 8 observations have been removed.


AlphaStats fragpipe24_LFQbench_umpire done

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

diann231_25fmol_vs_5fmol_rerun 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

Data columns: ['Protein.Group', 'Protein.Names', 'Genes', 'First.Protein.Description', 'N.Sequences_Intensity', 'N.Proteotypic.Sequences_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_01.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_02.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_03.mzML_Intensity', 'LFQ_Orbitrap_AIF_Condition_B_Sample_

## AlphaPeptStats UPS-Ecoli

In [14]:
imputation = 'randomforest'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'ups_ecoli' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        for pair_dr in listdir(path.join('./search_results', search_dr)) :
            if path.isdir(path.join('./search_results', search_dr, pair_dr)) and '_vs_' in pair_dr :
                conc_1, conc_2 = pair_dr.replace('fmol', '').split('_vs_')
                min_fc = 0.5*np.log2(min(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.')), 10))
                if search_dr.startswith('fragpipe') :
                    workdir = path.join('./search_results', search_dr, pair_dr, 'dia-quant-output')
                else :
                    workdir = path.join('./search_results', search_dr, pair_dr)
                # try :
                inpath = path.join(workdir, 'report.pg_matrix.tsv')
                if path.exists(inpath) :
                    colstarter = '/home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_' # 10fmol_inj2.mzML'
                    colsplitter = 'inj'
                    save_path = path.join(workdir, 'quant_alphastats_{}.tsv'.format(imputation))
                    try :
                        se = search_dr.split('ups_ecoli')[-1].split('_')[1]
                    except :
                        se = search_dr.split('_')[0]
                    de, h_de, e_de, y_de, lfq_df = alpha_quant(inpath, 
                                                                colstarter=colstarter, 
                                                                colsplitter=colsplitter, 
                                                                condstarter='',
                                                                organism_dict=ups_ecoli_organism_dict, 
                                                                imputation=imputation, 
                                                                min_fc=min_fc, 
                                                                search_engine=se, 
                                                                save_path=save_path)
                    print('AlphaStats', search_dr, 'done\n')
                else :
                    print('no input file in', search_dr)

2026-07-30 13:07:16,589 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-07-30 13:07:16,598 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-07-30 13:07:16,615 - root - INFO - Contaminations indicated in following columns: ['contamination_library', 'contamination_library'] were removed. In total 0 observations have been removed.
2026-07-30 13:07:16,617 - root - INFO - Imputing data...


diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

Data columns: ['Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes', 'First.Protein.Description', 'RD139_Wide_UPS1_5fmol_inj2.mzML_Intensity', 'RD139_Wide_UPS1_5fmol_inj1.mzML_Intensity', 'RD139_Wide_UPS1_0_25fmol_inj2.mzML_Intensity', 'RD139_Wide_UPS1_5fmol_inj3.mzML_Intensity', 'RD139_Wide_UPS1_0_25fmol_inj3.mzML_Intensity', 'RD139_Wide_UPS1_0_25fmol_inj1.mzML_Intensity', 'contamination_library']
Metadata sample names: ['RD139_Wide_UPS1_0_25fmol_inj1.mzML', 'RD139_Wide_UPS1_0_25fmol_inj2.mzML', 'RD139_Wide_UPS1_0_25fmol_inj3.mzML', 'RD139_Wide_UPS1_5fmol_inj1.mzML', 'RD139_Wide_UPS1_5fmol_inj2.mzML', 'RD139_Wide_UPS1_5fmol_inj3.mzML']
DataSet has been created.
[IterativeImputer] Completing matrix with shape (6, 1660)
[IterativeImputer] Change: 0.0, scaled tolerance: 1512350.0 
[IterativeImputer] Early stopping criterion reached.


2026-07-30 13:31:34,481 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-07-30 13:31:34,485 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-07-30 13:31:34,496 - root - INFO - Contaminations indicated in following columns: ['contamination_library', 'contamination_library'] were removed. In total 0 observations have been removed.
2026-07-30 13:31:34,498 - root - INFO - Imputing data...


AlphaStats fragpipe24_ups_ecoli_umpire done

Data columns: ['Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes', 'First.Protein.Description', 'RD139_Wide_UPS1_25fmol_inj1.mzML_Intensity', 'RD139_Wide_UPS1_25fmol_inj3.mzML_Intensity', 'RD139_Wide_UPS1_25fmol_inj2.mzML_Intensity', 'RD139_Wide_UPS1_2_5fmol_inj3.mzML_Intensity', 'RD139_Wide_UPS1_2_5fmol_inj1.mzML_Intensity', 'RD139_Wide_UPS1_2_5fmol_inj2.mzML_Intensity', 'contamination_library']
Metadata sample names: ['RD139_Wide_UPS1_2_5fmol_inj1.mzML', 'RD139_Wide_UPS1_2_5fmol_inj2.mzML', 'RD139_Wide_UPS1_2_5fmol_inj3.mzML', 'RD139_Wide_UPS1_25fmol_inj1.mzML', 'RD139_Wide_UPS1_25fmol_inj2.mzML', 'RD139_Wide_UPS1_25fmol_inj3.mzML']
DataSet has been created.
[IterativeImputer] Completing matrix with shape (6, 1665)


KeyboardInterrupt: 

In [ ]:
conc_1, conc_2 = pair_dr.replace('fmol', '').split('_vs_')
min_fc = 0.5*np.log2(min(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.')), 10))

## Manual UPS-Ecoli

In [44]:
imputation = 'min'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'ups_ecoli' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        for pair_dr in listdir(path.join('./search_results', search_dr)) :
            if path.isdir(path.join('./search_results', search_dr, pair_dr)) and '_vs_' in pair_dr :
                conc_1, conc_2 = pair_dr.replace('fmol', '').split('_vs_')
                min_fc = 0.5*np.log2(min(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.')), 10))
                if search_dr.startswith('fragpipe') :
                    workdir = path.join('./search_results', search_dr, pair_dr, 'dia-quant-output')
                else :
                    workdir = path.join('./search_results', search_dr, pair_dr)
                inpath = path.join(workdir, 'report.pg_matrix.tsv')
                if path.exists(inpath) :
                    colstarter = './input_files/PXD026600/RD139_Wide_UPS1_' # 10fmol_inj2.mzML'
                    colsplitter = 'inj'
                    save_path = path.join(workdir, 'quant_manual_{}.tsv'.format(imputation))
                    try :
                        se = search_dr.split('ups_ecoli')[-1].split('_')[1]
                    except :
                        se = search_dr.split('_')[0]
                    de, h_de, e_de, y_de, lfq_df = manual_quant(inpath, 
                                                                colstarter=colstarter, 
                                                                colsplitter=colsplitter, 
                                                                condstarter='',
                                                                organism_dict=ups_ecoli_organism_dict, 
                                                                imputation=imputation, 
                                                                min_fc=min_fc, 
                                                                search_engine=se, 
                                                                quant='manual',
                                                                save_path=save_path)
                    print('manual', search_dr, pair_dr, 'done\n')
                else :
                    print('no input file in', search_dr, pair_dr,)

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

['Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes', 'First.Protein.Description', '/home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_5fmol_inj2.mzML', '/home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_5fmol_inj1.mzML', '/home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_0_25fmol_inj2.mzML', '/home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_5fmol_inj3.mzML', '/home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_0_25fmol_inj3.mzML', '/home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_0_25fmol_inj1.mzML']
manual fragpipe24_ups_ecoli_umpire 5fmol_vs_0_25fmol done

['Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes', 'First.Protein.Description', '/home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_25fmol_inj1.mzML', '/home/lerost/DIA_tools_m

## Manual LFQbench orbitrap

In [69]:
imputation = 'min'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'LFQbench' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        if search_dr.startswith('fragpipe') :
            workdir = path.join('./search_results', search_dr, 'dia-quant-output')
        else :
            workdir = path.join('./search_results', search_dr)
        # try :
        inpath = path.join(workdir, 'report.pg_matrix.tsv')
        if path.exists(inpath) :
            colstarter = './input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_' # 10fmol_inj2.mzML'
            colsplitter = 'Sample'
            save_path = path.join(workdir, 'quant_manual_{}.tsv'.format(imputation))
            try :
                se = search_dr.split('LFQbench')[-1].split('_')[1]
            except :
                se = search_dr.split('_')[0]
            de, h_de, e_de, y_de, lfq_df = manual_quant(inpath, 
                                                        colstarter=colstarter, 
                                                        colsplitter=colsplitter, 
                                                        condstarter='Condition_',
                                                        organism_dict=lfqbench_organism_dict, 
                                                        imputation=imputation, 
                                                        max_nan=9,
                                                        min_fc=0.5, 
                                                        search_engine=se, 
                                                        quant='manual',
                                                        save_path=save_path)
            print('manual', search_dr, 'done\n')
        else :
            print('no input file in', search_dr)

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

['Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes', 'First.Protein.Description', '/home/lerost/DIA_tools_manuscript/input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_02.mzML', '/home/lerost/DIA_tools_manuscript/input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_02.mzML', '/home/lerost/DIA_tools_manuscript/input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_01.mzML', '/home/lerost/DIA_tools_manuscript/input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_01.mzML', '/home/lerost/DIA_tools_manuscript/input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_03.mzML', '/home/lerost/DIA_tools_manuscript/input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_03.mzML', '/home/lerost/DIA_tools_manuscript/input_files/LFQ

## Generate directLFQ intensities

In [ ]:
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
        # inpath = path.join(workdir, 'report.parquet')
        lfq_manager.run_lfq(inpath, columns_to_add=['Protein.Group', 'Protein.Ids'])

In [ ]:
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'ups_ecoli' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        for comp in listdir(path.join('./search_results', search_dr)) :
            if '_vs_' in comp : 
                if search_dr.startswith('fragpipe') :
                    inpath = path.join('./search_results', search_dr, comp, 'dia-quant-output', 'report.tsv')
                else :
                    inpath = path.join('./search_results', search_dr, comp, 'report.parquet')
                # inpath = path.join(workdir, 'report.parquet')
                lfq_manager.run_lfq(inpath, columns_to_add=['Protein.Group', 'Protein.Ids'])

## directLFQ p-value estimation UPS-Ecoli

In [60]:
conc_1 = '0_25'
conc_2 = '25'
0.5*np.log2(min(abs(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), 10)), 0.5*np.log2(10)


(-3.321928094887362, 1.660964047443681)

In [46]:
imputation = 'min'
quant = 'directlfq'

for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'ups_ecoli' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        for pair_dr in listdir(path.join('./search_results', search_dr)) :
            if path.isdir(path.join('./search_results', search_dr, pair_dr)) and '_vs_' in pair_dr :
                conc_1, conc_2 = pair_dr.replace('fmol', '').split('_vs_')
                min_fc = 0.5*np.log2(min(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.')), 10))
                if search_dr.startswith('fragpipe') :
                    inpath = path.join('./search_results', search_dr, pair_dr, 'dia-quant-output', 'report.tsv.protein_intensities.tsv')
                    prot_col=''
                else :
                    inpath = path.join('./search_results', search_dr, pair_dr, 'report.parquet.protein_intensities.tsv')
                    prot_col='protein'
                if path.exists(inpath) :
                    colstarter = 'RD139_Wide_UPS1_' # 10fmol_inj2.mzML'
                    colsplitter = 'inj'
                    save_path = path.join(path.dirname(inpath), 'quant_{}_{}.tsv'.format(quant, imputation))
                    try :
                        se = search_dr.split('ups_ecoli')[-1].split('_')[1]
                    except :
                        se = search_dr.split('_')[0]
                    de, h_de, e_de, y_de, lfq_df = manual_quant(inpath, 
                                                                colstarter=colstarter, 
                                                                colsplitter=colsplitter, 
                                                                condstarter='',
                                                                organism_dict=ups_ecoli_organism_dict, 
                                                                prot_col=prot_col,
                                                                imputation=imputation, 
                                                                min_fc=min_fc, 
                                                                search_engine=se, 
                                                                quant=quant,
                                                                save_path=save_path)
                    print('manual', search_dr, 'done\n')
                else :
                    print('no input file in', search_dr)

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

['protein', 'RD139_Wide_UPS1_0_25fmol_inj1', 'RD139_Wide_UPS1_0_25fmol_inj2', 'RD139_Wide_UPS1_0_25fmol_inj3', 'RD139_Wide_UPS1_5fmol_inj1', 'RD139_Wide_UPS1_5fmol_inj2', 'RD139_Wide_UPS1_5fmol_inj3', 'Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes']
manual fragpipe24_ups_ecoli_umpire done

['protein', 'RD139_Wide_UPS1_25fmol_inj1', 'RD139_Wide_UPS1_25fmol_inj2', 'RD139_Wide_UPS1_25fmol_inj3', 'RD139_Wide_UPS1_2_5fmol_inj1', 'RD139_Wide_UPS1_2_5fmol_inj2', 'RD139_Wide_UPS1_2_5fmol_inj3', 'Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes']
manual fragpipe24_ups_ecoli_umpire done

['protein', 'RD139_Wide_UPS1_0_1fmol_inj1', 'RD139_Wide_UPS1_0_1fmol_inj2', 'RD139_Wide_UPS1_0_1fmol_inj3', 'RD139_Wide_UPS1_2_5fmol_inj1', 'RD139_Wide_UPS1_2_5fmol_inj2', 'RD139_Wide_UPS1_2_5fmol_inj3', 'Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes']
manual fragpipe24_ups_ecoli_umpire done

['pr

## directLFQ p-value estimation LFQbench Orbitrap

In [47]:
imputation = 'min'
quant = 'directlfq'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'LFQbench' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv.protein_intensities.tsv')
            prot_col=''
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet.protein_intensities.tsv')
            prot_col=''
        if path.exists(inpath) :
            colstarter = 'LFQ_Orbitrap_AIF_Condition_' # 10fmol_inj2.mzML'
            colsplitter = 'Sample'
            save_path = path.join(path.dirname(inpath), 'quant_{}_{}.tsv'.format(quant, imputation))
            print(save_path)
            try :
                se = search_dr.split('LFQbench')[-1].split('_')[1]
            except :
                se = search_dr.split('_')[0]
            de, h_de, e_de, y_de, lfq_df = manual_quant(inpath, 
                                                        colstarter=colstarter, 
                                                        colsplitter=colsplitter, 
                                                        condstarter='Condition_',
                                                        organism_dict=lfqbench_organism_dict, 
                                                        prot_col=prot_col,
                                                        imputation=imputation, 
                                                        max_nan=9,
                                                        min_fc=0.5, 
                                                        search_engine=se, 
                                                        quant=quant,
                                                        save_path=save_path)
            print('manual', search_dr, 'done\n')
        else :
            print('no input file in', search_dr)

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

./search_results/fragpipe24_LFQbench_msfragger/dia-quant-output/quant_directlfq_min.tsv
['protein', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_01', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_02', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_03', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_01', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_02', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_03', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_01', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_02', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_03', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_01', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_02', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_03', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_01', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_02', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_03', 'LFQ_Orbitrap_AIF_Conditio

In [ ]:
tdf = pd.read_parquet('./search_results/diann231_LFQbench/report.parquet', )
ttdf = pd.read_csv('./search_results/diann231_LFQbench/report.parquet.protein_intensities.tsv', sep='\t')
exclude = ['Protein.Group', 'Protein.Ids', 'Protein.Ids_x', 'Protein.Ids_y']
cols = [col for col in ttdf.columns if not col in exclude]
ttdf = ttdf[cols].copy()
print(len(tdf), len(ttdf))
tdf = tdf.sort_values('Global.PG.Q.Value')[['Protein.Ids', 'Protein.Group']].drop_duplicates().drop_duplicates(subset=['Protein.Ids']).copy()
ttdf = ttdf.merge(tdf, how='left', left_on='protein', right_on='Protein.Ids')
print(len(tdf), len(ttdf))
ttdf = ttdf.dropna(subset=['Protein.Group'])
print(ttdf['Protein.Group'].isna().sum(), len(ttdf), len(ttdf[ttdf['protein'].map(lambda x: ';' in x)]), len(ttdf[(ttdf['protein'].map(lambda x: ';' in x)) & (ttdf['Protein.Group'].isna())]))
ttdf.head()
ttdf.to_csv('./search_results/diann231_LFQbench/report.parquet.protein_intensities.tsv', sep='\t', index=False)

## AlphaPeptStats LFQbench Bruker timsTOF

In [ ]:
imputation = 'randomforest'
# imputation = 'mean'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'LFQbench' in search_dr and 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        if search_dr.startswith('fragpipe') :
            workdir = path.join('./search_results', search_dr, 'dia-quant-output')
        else :
            workdir = path.join('./search_results', search_dr)
        # try :
        inpath = path.join(workdir, 'report.pg_matrix.tsv')
        if path.exists(inpath) :
            colstarter = './input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_' # 10fmol_inj2.mzML'
            colsplitter = 'Sample'
            save_path = path.join(workdir, 'quant_alphastats_{}.tsv'.format(imputation))
            try :
                se = search_dr.split('LFQbench')[-1].split('_')[1]
            except :
                se = search_dr.split('_')[0]
            de, h_de, e_de, y_de, lfq_df = alpha_quant(inpath, 
                                            colstarter=colstarter, 
                                            colsplitter=colsplitter, 
                                            condstarter='Condition_',
                                            organism_dict=lfqbench_organism_dict, 
                                            imputation=imputation, 
                                            # max_nan=9,
                                            min_fc=0.5, 
                                            search_engine=se, 
                                            # quant='manual',
                                            save_path=save_path)
            print('AlphaStats', search_dr, 'done\n')
        else :
            print('no input file in', search_dr)

2026-08-03 17:08:43,543 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-08-03 17:08:43,549 - root - INFO - Column 'contamination_library' has been added, to indicate contaminations.
The contaminant library was created by Frankenfield et al.:https://www.biorxiv.org/content/10.1101/2022.04.27.489766v2.full
2026-08-03 17:08:43,584 - root - INFO - Contaminations indicated in following columns: ['contamination_library', 'contamination_library'] were removed. In total 11 observations have been removed.
2026-08-03 17:08:43,586 - root - INFO - Imputing data...


diann231_LFQbenchTOF 

Data columns: ['Protein.Group', 'Protein.Names', 'Genes', 'First.Protein.Description', 'N.Sequences_Intensity', 'N.Proteotypic.Sequences_Intensity', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_01.d_Intensity', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_02.d_Intensity', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_03.d_Intensity', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_01.d_Intensity', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_02.d_Intensity', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_03.d_Intensity', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_01.d_Intensity', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_02.d_Intensity', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_03.d_Intensity', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_01.d_Intensity', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_02.d_Intensity', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_03.d_Intensity', 'LFQ_timsTOFPro_diaPASEF_Co

## Generate directLFQ intensities for timsTOF

In [61]:
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'LFQbench' in search_dr and 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
        # inpath = path.join(workdir, 'report.parquet')
        lfq_manager.run_lfq(inpath, columns_to_add=['Protein.Group', 'Protein.Ids'])

2026-08-03 16:50:18,255 - directlfq.lfq_manager - INFO - Starting directLFQ analysis.


diann231_LFQbenchTOF 



2026-08-03 16:50:18,573 - directlfq.utils - INFO - using input type diann_precursors
2026-08-03 16:50:22,472 - directlfq.lfq_manager - INFO - Performing sample normalization.
2026-08-03 16:50:25,149 - directlfq.lfq_manager - INFO - Estimating lfq intensities.
2026-08-03 16:50:25,160 - directlfq.protein_intensity_estimation - INFO - 15207 lfq-groups total
2026-08-03 16:50:35,450 - directlfq.protein_intensity_estimation - INFO - using 60 processes
2026-08-03 16:50:35,570 - directlfq.protein_intensity_estimation - INFO - lfq-object 0
2026-08-03 16:50:36,002 - directlfq.protein_intensity_estimation - INFO - lfq-object 200
2026-08-03 16:50:36,013 - directlfq.protein_intensity_estimation - INFO - lfq-object 100
2026-08-03 16:50:36,208 - directlfq.protein_intensity_estimation - INFO - lfq-object 700
2026-08-03 16:50:36,238 - directlfq.protein_intensity_estimation - INFO - lfq-object 500
2026-08-03 16:50:36,301 - directlfq.protein_intensity_estimation - INFO - lfq-object 300
2026-08-03 16:50:3

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 



2026-08-03 16:51:06,348 - directlfq.utils - INFO - using input type diann_precursors
2026-08-03 16:51:20,667 - directlfq.lfq_manager - INFO - Performing sample normalization.
2026-08-03 16:51:21,220 - directlfq.lfq_manager - INFO - Estimating lfq intensities.
2026-08-03 16:51:21,229 - directlfq.protein_intensity_estimation - INFO - 10432 lfq-groups total
2026-08-03 16:51:28,505 - directlfq.protein_intensity_estimation - INFO - using 60 processes
2026-08-03 16:51:28,601 - directlfq.protein_intensity_estimation - INFO - lfq-object 0
2026-08-03 16:51:29,023 - directlfq.protein_intensity_estimation - INFO - lfq-object 100
2026-08-03 16:51:29,178 - directlfq.protein_intensity_estimation - INFO - lfq-object 200
2026-08-03 16:51:29,229 - directlfq.protein_intensity_estimation - INFO - lfq-object 400
2026-08-03 16:51:29,492 - directlfq.protein_intensity_estimation - INFO - lfq-object 300
2026-08-03 16:51:29,507 - directlfq.protein_intensity_estimation - INFO - lfq-object 500
2026-08-03 16:51:2

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

fragpipe24_ups_ecoli_umpire_triqler 

fragpipe24_ups_ecoli_msfragger 

diann231_ups_ecoli 

fragpipe24_UPS-Ecoli.sh 



## directLFQ p-value estimation LFQbench timsTOF

In [66]:
imputation = 'min'
quant = 'directlfq'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'LFQbench' in search_dr and 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv.protein_intensities.tsv')
            prot_col=''
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet.protein_intensities.tsv')
            prot_col='protein'
        if path.exists(inpath) :
            colstarter = 'LFQ_timsTOFPro_diaPASEF_Condition_' # 10fmol_inj2.mzML'
            colsplitter = 'Sample'
            save_path = path.join(path.dirname(inpath), 'quant_{}_{}.tsv'.format(quant, imputation))
            print(save_path)
            try :
                se = search_dr.split('LFQbenchTOF')[-1].split('_')[1]
            except :
                se = search_dr.split('_')[0]
            print(inpath)
            de, h_de, e_de, y_de, lfq_df = manual_quant(inpath, 
                                                        colstarter=colstarter, 
                                                        colsplitter=colsplitter, 
                                                        condstarter='Condition_',
                                                        organism_dict=lfqbench_organism_dict, 
                                                        prot_col=prot_col,
                                                        imputation=imputation, 
                                                        max_nan=9,
                                                        min_fc=0.5, 
                                                        search_engine=se, 
                                                        quant=quant,
                                                        save_path=save_path)
            print('manual', search_dr, 'done\n')
        else :
            print('no input file in', search_dr)

diann231_LFQbenchTOF 

./search_results/diann231_LFQbenchTOF/quant_directlfq_min.tsv
./search_results/diann231_LFQbenchTOF/report.parquet.protein_intensities.tsv
['protein', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_01', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_02', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_01', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_02', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_01', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_02', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_03', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_02', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_03', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_02', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_03', 'LFQ_ti

## Manual LFQbench timsTOF

In [68]:
imputation = 'min'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    if 'LFQbench' in search_dr and 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and not 'triqler' in search_dr :
        if search_dr.startswith('fragpipe') :
            workdir = path.join('./search_results', search_dr, 'dia-quant-output')
        else :
            workdir = path.join('./search_results', search_dr)
        # try :
        inpath = path.join(workdir, 'report.pg_matrix.tsv')
        if path.exists(inpath) :
            colstarter = './input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_' # 10fmol_inj2.mzML'
            colsplitter = 'Sample'
            save_path = path.join(workdir, 'quant_manual_{}.tsv'.format(imputation))
            try :
                se = search_dr.split('LFQbench')[-1].split('_')[1]
            except :
                se = search_dr.split('_')[0]
            de, h_de, e_de, y_de, lfq_df = manual_quant(inpath, 
                                                        colstarter=colstarter, 
                                                        colsplitter=colsplitter, 
                                                        condstarter='Condition_',
                                                        organism_dict=lfqbench_organism_dict, 
                                                        imputation=imputation, 
                                                        max_nan=9,
                                                        min_fc=0.5, 
                                                        search_engine=se, 
                                                        quant='manual',
                                                        save_path=save_path)
            print('manual', search_dr, 'done\n')
        else :
            print('no input file in', search_dr)

diann231_LFQbenchTOF 

['Protein.Group', 'Protein.Names', 'Genes', 'First.Protein.Description', 'N.Sequences', 'N.Proteotypic.Sequences', '/home/lerost/DIA_tools_manuscript/input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_01.d', '/home/lerost/DIA_tools_manuscript/input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_02.d', '/home/lerost/DIA_tools_manuscript/input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_03.d', '/home/lerost/DIA_tools_manuscript/input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_01.d', '/home/lerost/DIA_tools_manuscript/input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_02.d', '/home/lerost/DIA_tools_manuscript/input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_03.d', '/home/lerost/DIA_tools_manuscript/input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_01.d', '/home/lerost/DIA_tools_m

## Copy output quant tables into quantitation directory

In [15]:
for dr in listdir('./search_results') :
    quants = 0
    if path.isdir(path.join('./search_results', dr)) : 
        if any([el.startswith('quant_') for el in listdir(path.join('./search_results', dr))]) :
            quants = sum([el.startswith('quant_') for el in listdir(path.join('./search_results', dr))])
            for el in listdir(path.join('./search_results', dr)) :
                if el.startswith('quant_') :
                    old_path = path.join('./search_results', dr, el)
                    new_path = path.join('./quantitation', dr + '_' + el)
                    !cp $old_path $new_path
            print( path.join('./search_results', dr), quants, sep='\t' )
        # else :
        #     print(1, path.join('./search_results', dr))
        elif path.isdir(path.join('./search_results', dr, 'dia-quant-output')) :
            if any([el.startswith('quant_') for el in listdir(path.join('./search_results', dr, 'dia-quant-output'))]) :
                quants = sum([el.startswith('quant_') for el in listdir(path.join('./search_results', dr, 'dia-quant-output'))])
                for el in listdir(path.join('./search_results', dr, 'dia-quant-output')) :
                    if el.startswith('quant_') :
                        old_path = path.join('./search_results', dr, 'dia-quant-output', el)
                        new_path = path.join('./quantitation', dr + '_' + el)
                        !cp $old_path $new_path
                print( path.join('./search_results', dr), quants, sep='\t' )
        elif 'ups_ecoli' in path.join('./search_results', dr) :
            for comp in listdir(path.join('./search_results', dr)) :
                quants = 0
                if path.isdir(path.join('./search_results', dr, comp, 'dia-quant-output')) :
                    if any([el.startswith('quant_') for el in listdir(path.join('./search_results', dr, comp, 'dia-quant-output'))]) :
                        quants = sum([el.startswith('quant_') for el in listdir(path.join('./search_results', dr, comp, 'dia-quant-output'))])
                        for el in listdir(path.join('./search_results', dr, comp, 'dia-quant-output')) :
                            if el.startswith('quant_') :
                                old_path = path.join('./search_results', dr, comp, 'dia-quant-output', el)
                                new_path = path.join('./quantitation', dr, comp + '_' + el)
                                !cp $old_path $new_path
                        print( path.join('./search_results', dr, comp), quants, sep='\t')
                    else :
                        print(2, path.join('./search_results', dr, comp))
                elif path.isdir(path.join('./search_results', dr, comp)) :
                    if any([el.startswith('quant_') for el in listdir(path.join('./search_results', dr, comp))]) :
                        quants = sum([el.startswith('quant_') for el in listdir(path.join('./search_results', dr, comp))])
                        for el in listdir(path.join('./search_results', dr, comp)) :
                            if el.startswith('quant_') :
                                old_path = path.join('./search_results', dr, comp, el)
                                new_path = path.join('./quantitation', dr, comp + '_' + el)
                                !cp $old_path $new_path
                        print( path.join('./search_results', dr, comp), quants, sep='\t')
                    else :
                        print(2, path.join('./search_results', dr, comp))
        else :
            print('error', path.join('./search_results', dr) )

./search_results/diann231_LFQbenchTOF	4
./search_results/fragpipe24_ups_ecoli_umpire/5fmol_vs_0_25fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/25fmol_vs_2_5fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/2_5fmol_vs_0_1fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/25fmol_vs_1fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/50fmol_vs_0_1fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/25fmol_vs_0_25fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/25fmol_vs_10fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/50fmol_vs_10fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/10fmol_vs_2_5fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/50fmol_vs_25fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/10fmol_vs_5fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/50fmol_vs_1fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/5fmol_vs_1fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/1fmol_vs_0_1fmol	4
./search_results/fragpipe24_ups_ecoli_umpire/5fmol_vs_2_5fmol